# Optuna Tuning LightGBM (TPESampler + Pruner)
Notebook ini melakukan hyperparameter tuning untuk target `luas_km2` menggunakan:
1. TPESampler
2. MedianPruner
3. Model LightGBM
4. Total 50 trial

In [11]:
from pathlib import Path
from time import perf_counter
import importlib
import json
import pickle
import subprocess
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [12]:
# Install package jika belum tersedia
def ensure_package(module_name, pip_name=None):
    pip_name = pip_name or module_name
    try:
        return importlib.import_module(module_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pip_name])
        return importlib.import_module(module_name)

optuna = ensure_package('optuna')
lgb = ensure_package('lightgbm')
ensure_package('optuna_integration', 'optuna-integration')

from optuna.integration import LightGBMPruningCallback

print('Optuna version:', optuna.__version__)
print('LightGBM version:', lgb.__version__)

Optuna version: 4.6.0
LightGBM version: 4.6.0


In [13]:
# Konfigurasi eksperimen
step_start = perf_counter()

RANDOM_STATE = 42
N_TRIALS = 50
TARGET_COL = 'luas_km2'

BASE_DIR = Path('.')
DATA_PATH = BASE_DIR / 'java-ash-hysplit-encoded.csv'
ARTIFACT_DIR = BASE_DIR / 'artifacts_optuna_luas_lightgbm'
MODEL_DIR = ARTIFACT_DIR / 'model'
METRIC_DIR = ARTIFACT_DIR / 'metrics'
TUNING_DIR = ARTIFACT_DIR / 'tuning'

for d in [ARTIFACT_DIR, MODEL_DIR, METRIC_DIR, TUNING_DIR]:
    d.mkdir(parents=True, exist_ok=True)

process_times = {}
process_times['setup'] = perf_counter() - step_start
print(f"Waktu setup: {process_times['setup']:.6f} detik")

Waktu setup: 0.002723 detik


In [14]:
# Load data
step_start = perf_counter()

df = pd.read_csv(DATA_PATH)

if TARGET_COL not in df.columns:
    raise ValueError(f'Target {TARGET_COL} tidak ditemukan di dataset.')

target_multi_cols = ['jarak_km', 'luas_km2', 'sudut_deg', 'radius_km']
feature_cols = [c for c in df.columns if c not in target_multi_cols]

X = df[feature_cols].copy()
y_raw = df[TARGET_COL].copy()
y = np.log1p(y_raw)

process_times['load_data'] = perf_counter() - step_start
print(f"Waktu load_data: {process_times['load_data']:.6f} detik")
print('Shape data:', df.shape)
print('Jumlah fitur:', len(feature_cols))

Waktu load_data: 0.012964 detik
Shape data: (1707, 19)
Jumlah fitur: 15


In [15]:
# Split data 80/10/10
step_start = perf_counter()

X_train, X_temp, y_train, y_temp, y_train_raw, y_temp_raw = train_test_split(
    X, y, y_raw, test_size=0.2, random_state=RANDOM_STATE
)

X_val, X_test, y_val, y_test, y_val_raw, y_test_raw = train_test_split(
    X_temp, y_temp, y_temp_raw, test_size=0.5, random_state=RANDOM_STATE
)

train_data = lgb.Dataset(X_train, label=y_train)
valid_data = lgb.Dataset(X_val, label=y_val)

process_times['split_data'] = perf_counter() - step_start
print(f"Waktu split_data: {process_times['split_data']:.6f} detik")
print('Train:', X_train.shape, '| Val:', X_val.shape, '| Test:', X_test.shape)

Waktu split_data: 0.004239 detik
Train: (1365, 15) | Val: (171, 15) | Test: (171, 15)


In [16]:
# Objective Optuna dengan TPESampler + Pruner
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        # 'boosting_type': 'gbdt',
        'verbosity': -1,
        'seed': RANDOM_STATE,
        'feature_fraction_seed': RANDOM_STATE,
        'bagging_seed': RANDOM_STATE,
        'data_random_seed': RANDOM_STATE,
        # Penting: nonaktifkan pre-filter agar min_data_in_leaf aman dituning dinamis
        'feature_pre_filter': False,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 16, 512),
        'max_depth': trial.suggest_int('max_depth', 3, 14),
        'min_data_in_leaf': trial.suggest_int('min_data_in_leaf', 5, 120),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.6, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.6, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 10),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 20.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 50.0, log=True),
        'min_split_gain': trial.suggest_float('min_split_gain', 0.0, 2.0),
    }

    callbacks = [
        lgb.early_stopping(stopping_rounds=200, verbose=False),
        LightGBMPruningCallback(trial, metric='rmse', valid_name='valid')
    ]

    booster = lgb.train(
        params=params,
        train_set=train_data,
        valid_sets=[valid_data],
        valid_names=['valid'],
        num_boost_round=5000,
        callbacks=callbacks
    )

    pred_val_log = booster.predict(X_val, num_iteration=booster.best_iteration)
    pred_val = np.expm1(pred_val_log)
    pred_val = np.clip(pred_val, a_min=0.0, a_max=None)

    rmse = np.sqrt(mean_squared_error(y_val_raw, pred_val))
    return rmse

In [17]:
# Jalankan tuning
step_start = perf_counter()

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE, multivariate=True)
pruner = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=100, interval_steps=50)
study = optuna.create_study(direction='minimize', sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

process_times['optuna_tuning'] = perf_counter() - step_start
print(f"Waktu optuna_tuning: {process_times['optuna_tuning']:.6f} detik")
print('Best RMSE (val):', study.best_value)
print('Best params:', study.best_params)

c:\Users\L E N O V O\AppData\Local\Programs\Python\Python311\Lib\site-packages\optuna\_experimental.py:32: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  warnings.warn(
[I 2026-03-14 17:58:02,263] A new study created in memory with name: no-name-b97f1d0b-4d3b-4505-8e7a-1c05e7101a34


  0%|          | 0/50 [00:00<?, ?it/s]

[I 2026-03-14 17:58:04,002] Trial 0 finished with value: 340.2175156429359 and parameters: {'learning_rate': 0.01184431975182039, 'num_leaves': 488, 'max_depth': 11, 'min_data_in_leaf': 74, 'feature_fraction': 0.6624074561769746, 'bagging_fraction': 0.662397808134481, 'bagging_freq': 1, 'lambda_l1': 1.1384928654456357, 'lambda_l2': 0.006763888939818983, 'min_split_gain': 1.416145155592091}. Best is trial 0 with value: 340.2175156429359.
[I 2026-03-14 17:58:05,012] Trial 1 finished with value: 306.84398276736084 and parameters: {'learning_rate': 0.005242693862597309, 'num_leaves': 498, 'max_depth': 12, 'min_data_in_leaf': 29, 'feature_fraction': 0.6727299868828402, 'bagging_fraction': 0.6733618039413735, 'bagging_freq': 4, 'lambda_l1': 0.0007599330121142788, 'lambda_l2': 0.00015467558930511852, 'min_split_gain': 0.5824582803960838}. Best is trial 1 with value: 306.84398276736084.
[I 2026-03-14 17:58:05,322] Trial 2 finished with value: 315.31135475918865 and parameters: {'learning_rate'

In [18]:
# Train final model dengan best params
step_start = perf_counter()

best_params = {
    **study.best_params,
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'verbosity': -1,
    'seed': RANDOM_STATE,
    'feature_fraction_seed': RANDOM_STATE,
    'bagging_seed': RANDOM_STATE,
    'data_random_seed': RANDOM_STATE,
    'feature_pre_filter': False,
}

best_model = lgb.train(
    params=best_params,
    train_set=train_data,
    valid_sets=[valid_data],
    valid_names=['valid'],
    num_boost_round=5000,
    callbacks=[lgb.early_stopping(stopping_rounds=200, verbose=False)]
)

pred_test_log = best_model.predict(X_test, num_iteration=best_model.best_iteration)
pred_test = np.expm1(pred_test_log)
pred_test = np.clip(pred_test, a_min=0.0, a_max=None)

test_mae = mean_absolute_error(y_test_raw, pred_test)
test_rmse = np.sqrt(mean_squared_error(y_test_raw, pred_test))
test_r2 = r2_score(y_test_raw, pred_test)

metrics = {
    'target': TARGET_COL,
    'test_MAE': float(test_mae),
    'test_RMSE': float(test_rmse),
    'test_R2': float(test_r2),
    'best_val_RMSE': float(study.best_value),
    'n_trials': int(N_TRIALS),
    'best_iteration': int(best_model.best_iteration),
    'n_pruned_trials': int(sum(t.state.name == 'PRUNED' for t in study.trials))
}

process_times['train_final_model'] = perf_counter() - step_start
print(f"Waktu train_final_model: {process_times['train_final_model']:.6f} detik")
print(metrics)

Waktu train_final_model: 0.252903 detik
{'target': 'luas_km2', 'test_MAE': 210.81560982095723, 'test_RMSE': 383.8723955069841, 'test_R2': 0.3318310786914692, 'best_val_RMSE': 275.48850460306886, 'n_trials': 50, 'best_iteration': 196, 'n_pruned_trials': 14}


In [19]:
# Simpan artefak
step_start = perf_counter()

best_model.save_model(str(MODEL_DIR / 'best_lgbm_luas_model.txt'))
with open(MODEL_DIR / 'best_lgbm_luas_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

with open(TUNING_DIR / 'best_params.json', 'w', encoding='utf-8') as f:
    json.dump(best_params, f, indent=2)

with open(METRIC_DIR / 'test_metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

trials_df = study.trials_dataframe()
trials_df.to_csv(TUNING_DIR / 'optuna_trials.csv', index=False)

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance_gain': best_model.feature_importance(importance_type='gain')
}).sort_values('importance_gain', ascending=False).reset_index(drop=True)
importance_df.to_csv(METRIC_DIR / 'feature_importance_gain.csv', index=False)

process_times['save_artifacts'] = perf_counter() - step_start
print(f"Waktu save_artifacts: {process_times['save_artifacts']:.6f} detik")
display(importance_df.head(15))

Waktu save_artifacts: 0.060807 detik


,feature,importance_gain
0,timestamp,1772.083383
1,tinggi_letusan_m,922.162554
2,kec_angin_km_jam,618.703788
3,arah_angin_deg,212.234831
4,duration,150.546211
5,amplitudo,119.616930
6,elevation,0.588017
7,alert_level,0.000000
8,latitude,0.000000
9,longitude,0.000000


In [20]:
# Ringkasan
metrics_df = pd.DataFrame([metrics])
timing_df = pd.DataFrame([
    {'process': k, 'duration_seconds': v} for k, v in process_times.items()
]).sort_values('duration_seconds', ascending=False).reset_index(drop=True)

timing_df.to_csv(METRIC_DIR / 'process_times.csv', index=False)

print('Artefak tersimpan di:', ARTIFACT_DIR)
display(metrics_df)
display(timing_df)

Artefak tersimpan di: artifacts_optuna_luas_lightgbm


,target,test_MAE,test_RMSE,test_R2,best_val_RMSE,n_trials,best_iteration,n_pruned_trials
0,luas_km2,210.81561,383.872396,0.331831,275.488505,50,196,14


,process,duration_seconds
0,optuna_tuning,11.867119
1,train_final_model,0.252903
2,save_artifacts,0.060807
3,load_data,0.012964
4,split_data,0.004239
5,setup,0.002723
